# Primary experimentation on DebateGPT human-human debates

This notebook analyzes only the human-human DebateGPT condition for primary results. Agent-generated debates are secondary qualitative material and are never used for H1-H6 ground truth.

In [1]:
from pathlib import Path
import importlib
import os
import sys
from getpass import getpass

root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src import data_loader as dl
from src import engine
importlib.reload(dl)
importlib.reload(engine)
DebateAnalysisPipeline = engine.DebateAnalysisPipeline

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Enter HF_TOKEN for this notebook session (input hidden): ')
if not os.environ['HF_TOKEN']:
    raise RuntimeError('HF_TOKEN is required for the prompted Llama analysis.')

In [2]:
debategpt_path = root / 'data' / 'raw' / 'debategpt'
rhetorical_repo = __import__('os').environ.get('RHETORICAL_MODEL_REPO')
debategpt_files = [p for p in debategpt_path.glob('*') if p.is_file() and p.suffix.lower() in {'.json', '.jsonl', '.csv'}]
if not debategpt_files:
    results = None
    print({'status': 'blocked', 'reason': 'DebateGPT export is missing', 'required_path': str(debategpt_path)})
else:
    try:
        debategpt = dl.load_debategpt(data_path=str(debategpt_path), checkpoint_dir=str(root / 'checkpoints'), force=False, human_human_only=True)
        transcripts = dl.build_transcripts(debategpt)
        pipeline = DebateAnalysisPipeline(backend='hf', checkpoint_dir=str(root / 'checkpoints'))
        results = pipeline.analyze_batch(transcripts)
        print({'status': 'completed', 'condition': 'human-human', 'transcripts': len(results), 'backend': 'hf', 'classifier_override': bool(rhetorical_repo)})
    except (OSError, RuntimeError, ValueError) as exc:
        results = None
        print({'status': 'blocked', 'reason': 'HF model initialization or inference failed', 'error_type': type(exc).__name__, 'error': str(exc), 'next_step': 'Verify HF_TOKEN access, CUDA/bitsandbytes compatibility, and available GPU memory, then rerun.'})


[09:00:22] INFO belief_debate_analyzer: Checkpoint hit: debategpt_v2__efa5031df430.pkl (skipping recompute)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[09:00:40] INFO sentence_transformers.base.model: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[09:00:43] INFO belief_debate_analyzer: Checkpoint hit: analysis__100.0__90ec0e168a34__r0s0.pkl (skipping recompute)
[09:00:43] INFO belief_debate_analyzer: Checkpoint hit: analysis__104.0__48e941238afe__r0s0.pkl (skipping recompute)
[09:00:43] INFO belief_debate_analyzer: Checkpoint hit: analysis__111.0__8a2862c249bf__r0s0.pkl (skipping recompute)
[09:00:43] INFO belief_debate_analyzer: Checkpoint hit: analysis__113.0__bf4ffacd6b60__r0s0.pkl (skipping recompute)
[09:00

{'status': 'completed', 'condition': 'human-human', 'transcripts': 150, 'backend': 'hf', 'classifier_override': False}


## Persist the real system output and derive H5/H6 inputs

Writes the completed human-human analysis to `data/processed/` for downstream notebooks. Nothing here is generated or estimated: every value is either an observed DebateGPT field or a value this run's models actually produced. H1-H4 still require human annotations and user-study data that do not exist yet, so `debategpt_instances.csv`, `convergence.csv` (H5), and `cw_por.csv` (H6) are the only artifacts this step can honestly produce.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src import data_loader as dl
from src.utils import CheckpointManager, stable_hash

if 'results' not in globals() or 'transcripts' not in globals() or 'debategpt' not in globals():
    debategpt_path = root / 'data' / 'raw' / 'debategpt'
    debategpt = dl.load_debategpt(
        data_path=str(debategpt_path),
        checkpoint_dir=str(root / 'checkpoints'),
        force=False,
        human_human_only=True,
    )
    transcripts = dl.build_transcripts(debategpt)
    ckpt = CheckpointManager(root / 'checkpoints')
    missing = []
    results = []
    for transcript in transcripts:
        content_key = stable_hash([t['text'] for t in transcript['turns']])
        key = f"analysis__{transcript['transcript_id']}__{content_key}__r0s0"
        if ckpt.exists(key, fmt='pkl'):
            results.append(ckpt.load(key, fmt='pkl'))
        else:
            missing.append(transcript['transcript_id'])
    if missing:
        results = None
        print({
            'status': 'blocked',
            'reason': 'analysis checkpoints are missing; run Cell 3 to compute them',
            'missing_transcript_count': len(missing),
            'missing_examples': missing[:5],
        })

if results is None:
    print({'status': 'blocked', 'reason': 'no completed analysis results to persist'})
else:
    processed_dir = root / 'data' / 'processed'
    processed_dir.mkdir(parents=True, exist_ok=True)
    transcripts_by_id = {t['transcript_id']: t for t in transcripts}

    system_output_path = processed_dir / 'debategpt_human_human_system_output.json'
    system_output_path.write_text(json.dumps(results, indent=2), encoding='utf-8')

    instances_cols = ['condition', 'participant_id', 'group_id', 'topic', 'turn', 'public_message',
                       'agreementPreTreatment', 'agreementPostTreatment',
                       'sideAgreementPreTreatment', 'sideAgreementPostTreatment']
    debategpt_instances = debategpt.rename(columns={'argument': 'public_message'})[instances_cols]
    debategpt_instances.to_csv(processed_dir / 'debategpt_instances.csv', index=False)

    convergence_rows = []
    for result in results:
        turns = transcripts_by_id[result['transcript_id']]['turns']
        per_speaker_idx = {}
        for index, speaker in enumerate(result['speakers']):
            per_speaker_idx.setdefault(speaker, []).append(index)
        for indexes in per_speaker_idx.values():
            assert all(turns[index]['agreement_pre'] == turns[indexes[0]]['agreement_pre'] for index in indexes)
            assert all(turns[index]['agreement_post'] == turns[indexes[0]]['agreement_post'] for index in indexes)
            convergence_rows.append({
                'system_pre': 0.0,
                'system_post': result['stance'][indexes[-1]]['S'],
                'agreementPreTreatment': turns[indexes[0]]['agreement_pre'],
                'agreementPostTreatment': turns[indexes[0]]['agreement_post'],
            })
    convergence = pd.DataFrame(convergence_rows)
    convergence.to_csv(processed_dir / 'convergence.csv', index=False)

    cw_por_rows = []
    for result in results:
        turns = transcripts_by_id[result['transcript_id']]['turns']
        for index, attribution in enumerate(result['attribution']):
            if attribution is None or attribution['label'] != 'strategic_persuasion':
                continue
            turn = turns[index]
            # paper §5.2: an *increase* in side agreement, not any change, with no proposition-agreement movement
            side_increased = turn['side_agreement_post'] > turn['side_agreement_pre']
            proposition_changed = turn['agreement_post'] != turn['agreement_pre']
            cw_por_rows.append({
                'normalized_jsd_weight': attribution['confidence'],
                'side_only_without_proposition_change': int(side_increased and not proposition_changed),
            })
    cw_por_df = pd.DataFrame(cw_por_rows)
    cw_por_path = processed_dir / 'cw_por.csv'
    if cw_por_df.empty:
        cw_por_path.unlink(missing_ok=True)
    else:
        cw_por_df.to_csv(cw_por_path, index=False)

    all_labels = [attribution['label'] for result in results for attribution in result['attribution'] if attribution is not None]
    all_quality = [quality for result in results for quality in result['quality']]
    summary = {
        'status': 'persisted',
        'transcripts': len(results),
        'participant_rows_in_convergence': len(convergence),
        'strategic_persuasion_turns_for_cw_por': len(cw_por_rows),
        'attribution_label_counts': pd.Series(all_labels).value_counts().to_dict(),
        'mean_quality': float(np.mean(all_quality)),
        'mean_abs_final_S': float(np.mean(convergence['system_post'].abs())),
        'artifacts': {
            'system_output': str(system_output_path),
            'debategpt_instances': str(processed_dir / 'debategpt_instances.csv'),
            'convergence': str(processed_dir / 'convergence.csv'),
            'cw_por': str(cw_por_path) if not cw_por_df.empty else None,
        },
    }
    if cw_por_df.empty:
        summary['cw_por_note'] = 'no strategic_persuasion turns were observed; cw_por.csv was not written because the downstream metric requires at least one observed row'
    print(summary)


[20:37:43] INFO belief_debate_analyzer: Checkpoint hit: debategpt_v2__efa5031df430.pkl (skipping recompute)


{'status': 'persisted', 'transcripts': 150, 'participant_rows_in_convergence': 300, 'strategic_persuasion_turns_for_cw_por': 3, 'attribution_label_counts': {'no_inflection': 390, 'echo': 306, 'evidence_adoption': 192, 'anchoring': 9, 'strategic_persuasion': 3}, 'mean_quality': 0.49333333333333335, 'mean_abs_final_S': 0.4616905615406526, 'artifacts': {'system_output': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/processed/debategpt_human_human_system_output.json', 'debategpt_instances': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/processed/debategpt_instances.csv', 'convergence': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/processed/convergence.csv', 'cw_por': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/processed/cw_por.csv'}}


In [5]:
import importlib

import src.metrics as metrics
import src.paper_metrics as paper_metrics

importlib.reload(paper_metrics)
importlib.reload(metrics)

if results is None:
    print({'status': 'blocked', 'reason': 'no completed analysis results are available'})
else:
    h5 = paper_metrics.pearson_nca_correlation(
        convergence[['system_pre', 'system_post']].to_dict(orient='records'),
        convergence[['agreementPreTreatment', 'agreementPostTreatment']].to_dict(orient='records'),
    )
    # paper §5.4 reporting commitment: state the preregistered verdict alongside the measurement,
    # not just the raw number, and never suppress a negative result.
    verdicts = metrics.check_falsification({'H5_nc_correlation_r': h5['r']})
    h5_verdict = verdicts.set_index('hypothesis').loc['H5_nc_correlation_r'].to_dict()

    cw_por_path = processed_dir / 'cw_por.csv'
    if cw_por_path.is_file() and cw_por_path.stat().st_size > 0:
        cw_por_records = pd.read_csv(cw_por_path).to_dict(orient='records')
        h6 = paper_metrics.cw_por(cw_por_records)
        h6_verdicts = metrics.check_falsification({'H6_cw_por_rate': h6['cw_por']})
        h6_verdict = h6_verdicts.set_index('hypothesis').loc['H6_cw_por_rate'].to_dict()
        h6_cw_por = {'status': 'completed', 'result': h6, 'falsification_verdict': h6_verdict}
    else:
        h6_cw_por = {
            'status': 'blocked',
            'reason': 'no strategic_persuasion turns were observed in this completed run',
            'required_input': str(cw_por_path),
        }

    evaluation_status = {
        'status': 'partial_evaluation_completed',
        'h5_nca_correlation': h5,
        'h5_falsification_verdict': h5_verdict,
        'h6_cw_por': h6_cw_por,
        'remaining_requirements': {
            'h1_h2_h4': str(root / 'data' / 'annotations' / 'human_labels.csv'),
            'h3': str(root / 'data' / 'user_study' / 'results.csv'),
        },
    }
    report_path = root / 'reports' / 'human_human_analysis_status.json'
    report_path.write_text(json.dumps(evaluation_status, indent=2) + '\n', encoding='utf-8')
    print({
        'status': evaluation_status['status'],
        'h5_r': h5['r'],
        'h5_p_value': h5['p_value'],
        'h5_n': h5['n'],
        'h5_verdict': h5_verdict['verdict'],
        'h6_status': h6_cw_por['status'],
        'h6_rate': h6_cw_por.get('result', {}).get('cw_por'),
        'h6_verdict': h6_cw_por.get('falsification_verdict', {}).get('verdict'),
        'report': str(report_path),
    })


{'status': 'partial_evaluation_completed', 'h5_r': 0.03228200471362814, 'h5_p_value': 0.5775604449098144, 'h5_n': 300, 'h5_verdict': 'FALSIFIED', 'h6_status': 'completed', 'h6_rate': 0.0, 'h6_verdict': 'FALSIFIED', 'report': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/human_human_analysis_status.json'}


## H4 ablations (real, partial)

The paper's H4 metric is an *accuracy drop* against human-annotated ground truth (H1/H2), which does not exist yet, so it stays blocked. What is computable now, from this run's real outputs with no fabrication: re-deriving attribution under each of the four ablation conditions (no rhetoric, no stance, no echo, no priority order) using the already-computed rhetoric/quality/stance values, plus freshly computed real semantic-similarity scores from the lightweight sentence-transformer (no Llama reload, no HF_TOKEN needed).


In [3]:
import gc
import importlib
import time

from src import engine
from src.utils import CheckpointManager
importlib.reload(engine)

TIME_BUDGET_SECONDS = 50  # stop and checkpoint well before the ~1-minute OOM window, resume on next run
SIMILARITY_CACHE_KEY = 'h4_similarity_cache_v1'

if results is None:
    print({'status': 'blocked', 'reason': 'no completed analysis results are available'})
else:
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    def _compute_pair_sims(result, similarity_model):
        texts = result['texts']
        speakers = result['speakers']
        last_own_idx = {}
        pair_sims = [None] * len(texts)
        for i, sp in enumerate(speakers):
            prev_own_idx = last_own_idx.get(sp)
            other_prev_idx = i - 1 if i > 0 else None
            sim_to_own_prior = similarity_model.similarity(texts[i], texts[prev_own_idx]) if prev_own_idx is not None else 0.0
            sim_to_other_prev = (
                similarity_model.similarity(texts[i], texts[other_prev_idx])
                if other_prev_idx is not None and speakers[other_prev_idx] != sp
                else 0.0
            )
            pair_sims[i] = (sim_to_own_prior, sim_to_other_prev)
            last_own_idx[sp] = i
        return pair_sims

    similarity_ckpt = CheckpointManager(root / 'checkpoints')
    # sim_to_own_prior / sim_to_other_prev are pure text comparisons, identical across every
    # ablation condition -- compute each transcript's pair once and persist under a time
    # budget, so a slow leak toward OOM never loses more than one short window of work.
    similarity_cache = similarity_ckpt.load(SIMILARITY_CACHE_KEY, fmt='pkl') if similarity_ckpt.exists(SIMILARITY_CACHE_KEY, fmt='pkl') else {}
    pending = [r for r in results if r['transcript_id'] not in similarity_cache]

    device_choice = 'cuda' if torch.cuda.is_available() else 'cpu'
    similarity = engine.SemanticSimilarity(backend='hf', device=device_choice)
    start_time = time.monotonic()
    processed_this_run = 0
    time_exceeded = False
    for result in pending:
        similarity_cache[result['transcript_id']] = _compute_pair_sims(result, similarity)
        processed_this_run += 1
        if time.monotonic() - start_time > TIME_BUDGET_SECONDS:
            time_exceeded = True
            break
    similarity_ckpt.save(SIMILARITY_CACHE_KEY, similarity_cache, fmt='pkl')
    del similarity
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    remaining = len(results) - len(similarity_cache)
    if remaining > 0:
        print({
            'status': 'partial_time_boxed_run',
            'device': device_choice,
            'processed_this_run': processed_this_run,
            'transcripts_cached': len(similarity_cache),
            'remaining': remaining,
            'time_exceeded': time_exceeded,
            'next_step': 'restart the kernel and re-run this cell; it resumes from the on-disk cache',
        })
    else:
        print({
            'status': 'similarity_cache_ready',
            'device': device_choice,
            'processed_this_run': processed_this_run,
            'transcripts_cached': len(similarity_cache),
            'total_transcripts': len(results),
        })

    def _classify_with_engine(result, attribution_engine, rhetoric_override=None):
        speakers = result['speakers']
        texts = result['texts']
        quality = result['quality']
        rhetoric = rhetoric_override if rhetoric_override is not None else result['rhetoric']
        stance_by_turn = result['stance']
        pair_sims = similarity_cache[result['transcript_id']]
        attributions = [None] * len(texts)
        last_own_idx = {}
        last_stance = {}
        for i, sp in enumerate(speakers):
            prev_own_idx = last_own_idx.get(sp)
            other_prev_idx = i - 1 if i > 0 else None
            sim_to_own_prior, sim_to_other_prev = pair_sims[i]
            stance_change = stance_by_turn[i]['S'] - last_stance.get(sp, 0.0)
            is_emp_causal = (rhetoric[i]['label'] in ('empirical', 'causal')) if rhetoric[i] else False
            attributions[i] = attribution_engine.classify_turn(
                current_rhetoric=rhetoric[i]['probs'] if rhetoric[i] else {l: 0.25 for l in engine.RHETORIC_LABELS},
                prev_rhetoric=(rhetoric[prev_own_idx]['probs'] if prev_own_idx is not None and rhetoric[prev_own_idx] else None),
                other_agent_prev_rhetoric=(rhetoric[other_prev_idx]['probs'] if other_prev_idx is not None and rhetoric[other_prev_idx] else None),
                quality_score=quality[i],
                stance_change=stance_change,
                prev_stance=last_stance.get(sp, 0.0),
                sim_to_own_prior=sim_to_own_prior,
                sim_to_other_prev=sim_to_other_prev,
                is_empirical_or_causal=is_emp_causal,
            )
            last_own_idx[sp] = i
            last_stance[sp] = stance_by_turn[i]['S']
        return attributions

    if remaining == 0:
        ablation_engines = {
            'no_rhetoric': (engine.AttributionEngine(disable_rhetoric=True), True),
            'no_stance': (engine.AttributionEngine(disable_stance=True), False),
            'no_echo': (engine.AttributionEngine(disable_echo=True), False),
        }

        ablation_label_counts = {}
        for name, (attribution_engine, strip_rhetoric) in ablation_engines.items():
            labels = []
            for result in results:
                rhetoric_override = [None] * len(result['texts']) if strip_rhetoric else None
                attributions = _classify_with_engine(result, attribution_engine, rhetoric_override)
                labels.extend(a['label'] for a in attributions)
            ablation_label_counts[name] = pd.Series(labels).value_counts().to_dict()

        ablation_label_counts['full'] = pd.Series(
            [a['label'] for r in results for a in r['attribution'] if a is not None]
        ).value_counts().to_dict()

        # no-priority-order ablation needs no recomputation: `signals` already records every
        # signal's independent verdict, regardless of which one the priority order surfaced.
        all_signals = [a['signals'] for r in results for a in r['attribution'] if a is not None]
        signal_firing_rate = {
            signal: float(np.mean([s[signal] for s in all_signals]))
            for signal in engine.AttributionEngine.PRIORITY
        }

        h4_status = {
            'status': 'partial_real_ablation_outputs',
            'note': "Attribution accuracy deltas (the paper's H4 metric) require H1/H2 human ground "
                    'truth, which does not exist yet. These are the real, computed attribution label '
                    'distributions under each ablation, and the per-signal firing rate for the '
                    'no-priority-order condition -- not accuracy.',
            'attribution_label_counts_by_condition': ablation_label_counts,
            'no_priority_order_signal_firing_rate': signal_firing_rate,
            'blocked_on': str(root / 'data' / 'annotations' / 'human_labels.csv'),
        }
        h4_report_path = root / 'reports' / 'h4_ablation_status.json'
        h4_report_path.write_text(json.dumps(h4_status, indent=2) + '\n', encoding='utf-8')
        print({
            'status': h4_status['status'],
            'attribution_label_counts_by_condition': ablation_label_counts,
            'no_priority_order_signal_firing_rate': signal_firing_rate,
            'report': str(h4_report_path),
        })



[20:39:37] INFO sentence_transformers.base.model: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{'status': 'similarity_cache_ready', 'device': 'cuda', 'processed_this_run': 0, 'transcripts_cached': 150, 'total_transcripts': 150}
{'status': 'partial_real_ablation_outputs', 'attribution_label_counts_by_condition': {'no_rhetoric': {'no_inflection': 605, 'evidence_adoption': 204, 'echo': 68, 'anchoring': 23}, 'no_stance': {'no_inflection': 393, 'echo': 306, 'evidence_adoption': 192, 'anchoring': 9}, 'no_echo': {'no_inflection': 665, 'evidence_adoption': 192, 'anchoring': 40, 'strategic_persuasion': 3}, 'full': {'no_inflection': 390, 'echo': 306, 'evidence_adoption': 192, 'anchoring': 9, 'strategic_persuasion': 3}}, 'no_priority_order_signal_firing_rate': {'evidence_adoption': 0.21333333333333335, 'strategic_persuasion': 0.0033333333333333335, 'echo': 0.41888888888888887, 'anchoring': 0.044444444444444446}, 'report': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/h4_ablation_status.json'}


/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: divide by zero encountered in log2
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in multiply
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in divide
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: divide by zero encountered in log2
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in multiply
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pa

## H1 / H2 (real, from returned human annotations)

The three annotators (TS, GS, RK) have returned their completed 145-instance packets. `annotation_GS_marked.csv` arrived with literal backslash-escaping corruption (artifact of a copy/paste through a markdown-escaping tool) and is not used; `annotation_GS_marked_clean.csv` is the cleaned, validated equivalent used here. All three files were validated beforehand: 145/145 rows, 0 empty/invalid labels, 0 duplicate instance ids, full id coverage matching the master sample; TS additionally supplied a 145-entry rationale file cross-checked against her own labels with 0 mismatches.

This cell builds the real long-format `data/annotations/human_labels.csv`, computes H1 (Fleiss' kappa across every label category actually used -- including `ambiguous`, which is a legitimate non-forced-choice outcome, not a defined attribution category), derives a majority label per instance (>=2 of 3 agreeing; 3-way splits are marked `no_majority`), and computes H2 (system attribution accuracy / per-class F1 / confusion matrix / joint-vs-divergent split) against the system's real last-turn attribution label for each labeled participant. Instances whose majority label is `ambiguous` or `no_majority` are excluded from H2 and reported separately, since the system has no corresponding category to be scored against.


In [2]:
annotation_dir = root / 'data' / 'annotations'
annotator_files = {
    'TS': annotation_dir / 'annotation_TS_marked.csv',
    'GS': annotation_dir / 'annotation_GS_marked_clean.csv',
    'RK': annotation_dir / 'annotation_RK_marked.csv',
}
master_path = annotation_dir / 'annotation_sample_master.csv'
missing_inputs = [str(p) for p in list(annotator_files.values()) + [master_path] if not p.exists()]

if results is None:
    print({'status': 'blocked', 'reason': 'no completed analysis results are available'})
elif missing_inputs:
    print({'status': 'blocked', 'reason': 'human annotation files are missing', 'missing': missing_inputs})
else:
    from src.metrics import (
        ATTRIBUTION_LABELS,
        attribution_accuracy,
        accuracy_by_movement,
        check_falsification,
        fleiss_kappa,
        ratings_from_long_df,
    )

    per_annotator_counts = {}
    long_rows = []
    for name, path in annotator_files.items():
        adf = pd.read_csv(path, comment='#')
        per_annotator_counts[name] = adf['label'].value_counts().to_dict()
        long_rows.append(adf[['instance_id', 'annotator', 'label']])
    human_labels = pd.concat(long_rows, ignore_index=True)
    human_labels_path = annotation_dir / 'human_labels.csv'
    human_labels.to_csv(human_labels_path, index=False)

    # H1: Fleiss' kappa across every label category actually used.
    ratings = ratings_from_long_df(human_labels, 'instance_id', 'annotator', 'label')
    labels_used = sorted(human_labels['label'].unique())
    kappa = fleiss_kappa(ratings)

    def _majority(labels):
        counts = labels.value_counts()
        top = counts.iloc[0]
        winners = counts[counts == top].index.tolist()
        return winners[0] if len(winners) == 1 and top >= 2 else 'no_majority'

    majority = human_labels.groupby('instance_id')['label'].apply(_majority).rename('majority_label')
    master = pd.read_csv(master_path).merge(majority, on='instance_id')

    by_transcript = {r['transcript_id']: r for r in results}

    def _system_label(row):
        r = by_transcript.get(str(row['group_id']))
        if r is None:
            return None
        idxs = [i for i, sp in enumerate(r['speakers']) if sp == row['participant_id']]
        if not idxs:
            return None
        return r['attribution'][idxs[-1]]['label']

    master['system_label'] = master.apply(_system_label, axis=1)

    eligible = master[master['majority_label'].isin(ATTRIBUTION_LABELS)].copy()
    excluded = master[~master['majority_label'].isin(ATTRIBUTION_LABELS)]

    h2 = attribution_accuracy(eligible['majority_label'], eligible['system_label'], labels=ATTRIBUTION_LABELS)
    by_movement = accuracy_by_movement(eligible, 'majority_label', 'system_label', 'movement_class')

    verdicts = check_falsification({
        'H1_fleiss_kappa': kappa,
        'H2_overall_accuracy': h2['overall_accuracy'],
        'H2_f1_evidence_adoption': h2['per_class_f1']['evidence_adoption'],
        'H2_f1_anchoring': h2['per_class_f1']['anchoring'],
        'H2_f1_echo': h2['per_class_f1']['echo'],
        'H2_f1_strategic_persuasion': h2['per_class_f1']['strategic_persuasion'],
    }).set_index('hypothesis')['verdict'].to_dict()

    h1_h2_status = {
        'status': 'computed_real',
        'n_instances_labeled': int(len(master)),
        'per_annotator_label_counts': per_annotator_counts,
        'H1_fleiss_kappa': {
            'labels_used': labels_used,
            'observed': kappa,
            'verdict': verdicts['H1_fleiss_kappa'],
        },
        'majority_label_distribution': majority.value_counts().to_dict(),
        'H2': {
            'n_eligible_instances': int(len(eligible)),
            'n_excluded_ambiguous_or_no_majority': int(len(excluded)),
            'excluded_majority_label_distribution': excluded['majority_label'].value_counts().to_dict(),
            'overall_accuracy': h2['overall_accuracy'],
            'macro_f1': h2['macro_f1'],
            'per_class_f1': h2['per_class_f1'],
            'confusion_matrix': h2['confusion_matrix'].tolist(),
            'confusion_matrix_labels': h2['labels'],
            'accuracy_by_movement_class': by_movement,
            'verdicts': {k: verdicts[k] for k in [
                'H2_overall_accuracy', 'H2_f1_evidence_adoption', 'H2_f1_anchoring',
                'H2_f1_echo', 'H2_f1_strategic_persuasion',
            ]},
        },
        'human_labels_path': str(human_labels_path),
    }
    h1_h2_report_path = root / 'reports' / 'h1_h2_status.json'
    h1_h2_report_path.write_text(json.dumps(h1_h2_status, indent=2) + '\n', encoding='utf-8')
    print({
        'per_annotator_label_counts': per_annotator_counts,
        'H1_fleiss_kappa': h1_h2_status['H1_fleiss_kappa'],
        'majority_label_distribution': h1_h2_status['majority_label_distribution'],
        'H2_n_eligible': len(eligible),
        'H2_n_excluded': len(excluded),
        'H2_overall_accuracy': h2['overall_accuracy'],
        'H2_per_class_f1': h2['per_class_f1'],
        'H2_accuracy_by_movement_class': by_movement,
        'report': str(h1_h2_report_path),
    })


{'per_annotator_label_counts': {'TS': {'ambiguous': 100, 'anchoring': 36, 'evidence_adoption': 7, 'echo': 2}, 'GS': {'anchoring': 134, 'evidence_adoption': 4, 'ambiguous': 3, 'strategic_persuasion': 2, 'echo': 2}, 'RK': {'ambiguous': 137, 'evidence_adoption': 3, 'anchoring': 2, 'strategic_persuasion': 2, 'echo': 1}}, 'H1_fleiss_kappa': {'labels_used': ['ambiguous', 'anchoring', 'echo', 'evidence_adoption', 'strategic_persuasion'], 'observed': -0.2776020588581981, 'verdict': 'FALSIFIED'}, 'majority_label_distribution': {'ambiguous': 97, 'anchoring': 37, 'no_majority': 11}, 'H2_n_eligible': 37, 'H2_n_excluded': 108, 'H2_overall_accuracy': 0.10810810810810811, 'H2_per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.1951219512195122, 'echo': 0.0, 'strategic_persuasion': 0.0}, 'H2_accuracy_by_movement_class': {'divergent_movement': {'accuracy': 0.0, 'n': 17}, 'joint_movement': {'accuracy': 0.2, 'n': 20}}, 'report': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/

## H4 (real, now unblocked by H1/H2 ground truth)

H1/H2 produced a real, labeled ground-truth subset (`eligible`, n=37: the instances with a genuine 2-of-3 human majority on one of the four defined attribution categories). This cell re-scores that same subset under each ablation condition (no rhetoric, no stance, no echo) using the participant's last-turn attribution label, and reports the accuracy change versus the full-pipeline condition. The paper's only falsifiable H4 metric is the no-rhetoric accuracy drop; the no-stance/no-echo drops are reported for completeness but have no separate preregistered threshold.


In [4]:
if results is None or 'eligible' not in globals():
    print({'status': 'blocked', 'reason': 'requires the H1/H2 cell (eligible ground-truth subset) and the H4 ablation cell to run first'})
else:
    def _last_turn_labels(attribution_engine, strip_rhetoric):
        by_result = {}
        for result in results:
            rhetoric_override = [None] * len(result['texts']) if strip_rhetoric else None
            by_result[result['transcript_id']] = _classify_with_engine(result, attribution_engine, rhetoric_override)
        return by_result

    def _label_for_row(row, by_result):
        transcript_key = str(row['group_id'])
        r = by_transcript[transcript_key]
        idxs = [i for i, sp in enumerate(r['speakers']) if sp == row['participant_id']]
        return by_result[transcript_key][idxs[-1]]['label']

    h4_accuracy_by_condition = {
        'full': attribution_accuracy(eligible['majority_label'], eligible['system_label'], labels=ATTRIBUTION_LABELS),
    }
    for name, (attribution_engine, strip_rhetoric) in ablation_engines.items():
        by_result = _last_turn_labels(attribution_engine, strip_rhetoric)
        pred_labels = eligible.apply(lambda row: _label_for_row(row, by_result), axis=1)
        h4_accuracy_by_condition[name] = attribution_accuracy(eligible['majority_label'], pred_labels, labels=ATTRIBUTION_LABELS)

    full_accuracy = h4_accuracy_by_condition['full']['overall_accuracy']
    accuracy_drop_pts = {
        name: round((full_accuracy - stats['overall_accuracy']) * 100, 2)
        for name, stats in h4_accuracy_by_condition.items() if name != 'full'
    }

    h4_verdicts = check_falsification({
        'H4_ablation_rhetoric_drop_pts': accuracy_drop_pts['no_rhetoric'],
    }).set_index('hypothesis')['verdict'].to_dict()

    h4_real_status = {
        'status': 'computed_real',
        'n_eligible_instances': int(len(eligible)),
        'accuracy_by_condition': {
            name: {'overall_accuracy': stats['overall_accuracy'], 'macro_f1': stats['macro_f1'], 'per_class_f1': stats['per_class_f1']}
            for name, stats in h4_accuracy_by_condition.items()
        },
        'accuracy_drop_pts_vs_full': accuracy_drop_pts,
        'H4_ablation_rhetoric_drop_pts_verdict': h4_verdicts['H4_ablation_rhetoric_drop_pts'],
        'note': 'accuracy_drop_pts_vs_full is full_accuracy - ablation_accuracy in percentage points; '
                'a negative value means removing that signal increased accuracy on this ground-truth subset.',
    }
    h4_real_report_path = root / 'reports' / 'h4_ablation_status.json'
    existing_h4 = json.loads(h4_real_report_path.read_text(encoding='utf-8')) if h4_real_report_path.is_file() else {}
    existing_h4['real_accuracy_drop'] = h4_real_status
    h4_real_report_path.write_text(json.dumps(existing_h4, indent=2) + '\n', encoding='utf-8')
    print({
        'accuracy_by_condition': h4_real_status['accuracy_by_condition'],
        'accuracy_drop_pts_vs_full': accuracy_drop_pts,
        'H4_ablation_rhetoric_drop_pts_verdict': h4_verdicts['H4_ablation_rhetoric_drop_pts'],
        'report': str(h4_real_report_path),
    })


{'accuracy_by_condition': {'full': {'overall_accuracy': 0.10810810810810811, 'macro_f1': 0.04878048780487805, 'per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.1951219512195122, 'echo': 0.0, 'strategic_persuasion': 0.0}}, 'no_rhetoric': {'overall_accuracy': 0.13513513513513514, 'macro_f1': 0.05952380952380952, 'per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.23809523809523808, 'echo': 0.0, 'strategic_persuasion': 0.0}}, 'no_stance': {'overall_accuracy': 0.10810810810810811, 'macro_f1': 0.04878048780487805, 'per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.1951219512195122, 'echo': 0.0, 'strategic_persuasion': 0.0}}, 'no_echo': {'overall_accuracy': 0.13513513513513514, 'macro_f1': 0.05952380952380952, 'per_class_f1': {'evidence_adoption': 0.0, 'anchoring': 0.23809523809523808, 'echo': 0.0, 'strategic_persuasion': 0.0}}}, 'accuracy_drop_pts_vs_full': {'no_rhetoric': -2.7, 'no_stance': 0.0, 'no_echo': -2.7}, 'H4_ablation_rhetoric_drop_pts_verdict': 'FALSIFIED', 'r

/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: divide by zero encountered in log2
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in multiply
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in divide
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: divide by zero encountered in log2
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/src/engine.py:111: RuntimeWarning: invalid value encountered in multiply
  return float(np.sum(np.where(x > 0, x * np.log2(x / y), 0.0)))
/home/pa